# ViRAGE: демонстрация сервисов по реальной цепочке

Формат: **одна ячейка = один сервис/компонент**.

В ячейке создаются минимальные входные данные, вызывается реальный `invoke(...)`, а последняя строка возвращает объект или словарь для отображения в Jupyter.

Notebook использует реальные модели из конфигурации проекта: Ollama/OpenAI не подменяются mock-объектами.


In [8]:
import copy
import sys
from datetime import datetime
from pathlib import Path

from rich import print as rprint

# Если notebook лежит не в корне проекта, укажи путь явно:
PROJECT_ROOT = Path(r".").resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"D:/programming/projects/ViRAGE").resolve()

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Не найдена папка src в PROJECT_ROOT={PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "ui" / "config" / "project-gemma4.toml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "ui" / "config" / "project-gemma4.toml"

RUN_ID = "manual_service_chain_" + datetime.now().strftime("%Y%m%d_%H%M%S")
QUERY = "Show average sales over time by region."
USER_CONTEXT = {}

from src.application.bootstrap import bootstrap_project_environment
from src.application.project_config import load_project_config
from src.infrastructure.runtime import RuntimeContext
from src.llm.factory import build_chat_model

bootstrap_project_environment()
config = load_project_config(CONFIG_PATH)
settings = config.settings.model_copy(deep=True)
settings.artifact_root = PROJECT_ROOT / "artifacts" / "manual_service_chain"
settings.visrag_corpus_root = PROJECT_ROOT / "rag_corpus" / "runtime"
settings.spec_generation_backend = "vegachat_codegen"
settings.data_profile_sample_size = 5


runtime = RuntimeContext(
    settings=settings,
    reasoning_llm=build_chat_model(config.reasoning_model),
    spec_llm=build_chat_model(config.spec_model),
    vlm=build_chat_model(config.vlm_model),
    vision_judge_llm=build_chat_model(config.vision_judge_model),
)
runtime.current_run_id = RUN_ID
runtime.reset_artifact_indices(run_id=RUN_ID)
runtime.ensure_run_dir(RUN_ID)


def as_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if isinstance(obj, Path):
        return obj.as_posix()
    if isinstance(obj, dict):
        return {k: as_dict(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [as_dict(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(as_dict(v) for v in obj)
    return obj


{
    "project_root": PROJECT_ROOT.as_posix(),
    "config_path": CONFIG_PATH.as_posix(),
    "run_id": RUN_ID,
    "query": QUERY,
    "artifact_root": settings.artifact_root.as_posix(),
    "reasoning_model": config.reasoning_model.model,
    "spec_model": config.spec_model.model,
    "vlm_model": config.vlm_model.model,
    "vision_judge_model": config.vision_judge_model.model,
}

{'project_root': 'D:/programming/projects/ViRAGE',
 'config_path': 'D:/programming/projects/ViRAGE/ui/config/project-gemma4.toml',
 'run_id': 'manual_service_chain_20260524_134601',
 'query': 'Show average sales over time by region.',
 'artifact_root': 'D:/programming/projects/ViRAGE/artifacts/manual_service_chain',
 'reasoning_model': 'gemma4:31b-cloud',
 'spec_model': 'gemma4:31b-cloud',
 'vlm_model': 'gemma4:31b-cloud',
 'vision_judge_model': 'gemma4:31b-cloud'}

## Входные данные

In [9]:
import pandas as pd

DATA_PATH = PROJECT_ROOT / "artifacts" / "manual_service_chain" / "demo_sales.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

source_df = pd.DataFrame({
    "Order Date": pd.date_range("2024-01-01", periods=12, freq="MS"),
    "Region": ["North", "South", "East", "West"] * 3,
    "Sales": [1200, 900, 1500, 700, 1350, 980, 1620, 860, 1420, 1040, 1710, 940],
    "Profit": [220, 140, 310, 80, 260, 170, 340, 110, 290, 190, 380, 130],
    "Discount": [0.05, 0.10, 0.03, 0.15, 0.07, 0.11, 0.02, 0.13, 0.06, 0.09, 0.01, 0.12],
})
source_df.to_csv(DATA_PATH, index=False)
source_df


,Order Date,Region,Sales,Profit,Discount
0,2024-01-01,North,1200,220,0.05
1,2024-02-01,South,900,140,0.10
2,2024-03-01,East,1500,310,0.03
3,2024-04-01,West,700,80,0.15
4,2024-05-01,North,1350,260,0.07
5,2024-06-01,South,980,170,0.11
6,2024-07-01,East,1620,340,0.02
7,2024-08-01,West,860,110,0.13
8,2024-09-01,North,1420,290,0.06
9,2024-10-01,South,1040,190,0.09


## 1. DataProfilerService

In [10]:
from src.services.data_profiler import DataProfilerService

data_profile = DataProfilerService().invoke(DATA_PATH.as_posix(), runtime=runtime)
rprint(as_dict(data_profile))


{
    'row_count': 12,
    'col_count': 5,
    'columns': [
        {
            'name': 'Order Date',
            'dtype': 'datetime',
            'role': 'temporal',
            'missing_ratio': 0.0,
            'unique_count': 12,
            'safe_name': 'Order_Date',
            'min_value': '2024-01-01T00:00:00',
            'max_value': '2024-12-01T00:00:00',
            'sample_values': ['2024-11-01', '2024-10-01', '2024-01-01', '2024-09-01', '2024-06-01'],
            'outlier_count': 0,
            'outlier_ratio': 0.0,
            'is_identifier': False,
            'is_high_cardinality': False,
            'raw_dtype': 'object',
            'missing_like_ratio': 0.0,
            'quality_flags': ['temporal_like', 'good_for_temporal_axis'],
            'preparation_hints': ['normalize_temporal_values']
        },
        {
            'name': 'Region',
            'dtype': 'categorical',
            'role': 'dimension',
            'missing_ratio': 0.0,
            'unique_count': 4,
            'safe_name': 'Region',
            'min_value': None,
            'max_value': None,
            'sample_values': ['East', 'South', 'North', 'North', 'South'],
            'outlier_count': 0,
            'outlier_ratio': 0.0,
            'is_identifier': False,
            'is_high_cardinality': False,
            'raw_dtype': 'object',
            'missing_like_ratio': 0.0,
            'quality_flags': ['good_for_grouping', 'safe_for_color'],
            'preparation_hints': []
        },
        {
            'name': 'Sales',
            'dtype': 'numeric',
            'role': 'measure',
            'missing_ratio': 0.0,
            'unique_count': 12,
            'safe_name': 'Sales',
            'min_value': 700.0,
            'max_value': 1710.0,
            'sample_values': [1710, 1040, 1200, 1420, 980],
            'outlier_count': 0,
            'outlier_ratio': 0.0,
            'is_identifier': False,
            'is_high_cardinality': False,
            'raw_dtype': 'int64',
            'missing_like_ratio': 0.0,
            'quality_flags': ['good_for_measure'],
            'preparation_hints': []
        },
        {
            'name': 'Profit',
            'dtype': 'numeric',
            'role': 'measure',
            'missing_ratio': 0.0,
            'unique_count': 12,
            'safe_name': 'Profit',
            'min_value': 80.0,
            'max_value': 380.0,
            'sample_values': [380, 190, 220, 290, 170],
            'outlier_count': 0,
            'outlier_ratio': 0.0,
            'is_identifier': False,
            'is_high_cardinality': False,
            'raw_dtype': 'int64',
            'missing_like_ratio': 0.0,
            'quality_flags': ['good_for_measure'],
            'preparation_hints': []
        },
        {
            'name': 'Discount',
            'dtype': 'numeric',
            'role': 'measure',
            'missing_ratio': 0.0,
            'unique_count': 12,
            'safe_name': 'Discount',
            'min_value': 0.01,
            'max_value': 0.15,
            'sample_values': [0.01, 0.09, 0.05, 0.06, 0.11],
            'outlier_count': 0,
            'outlier_ratio': 0.0,
            'is_identifier': False,
            'is_high_cardinality': False,
            'raw_dtype': 'float64',
            'missing_like_ratio': 0.0,
            'quality_flags': ['good_for_measure'],
            'preparation_hints': []
        }
    ],
    'quality_notes': ['Detected time-like columns: Order Date.'],
    'complexity_hints': [],
    'data_complexity': 'standard',
    'profile_status': 'ok',
    'errors': [],
    'sample_strategy': 'random',
    'sample_seed': 42,
    'sample_size': 5,
    'source_format': 'csv',
    'source_encoding': 'utf-8'
}

## 2. DataProfilePromptFormatter

In [11]:
from src.services.data_profile_prompt_formatter import DataProfilePromptFormatter

data_profile_prompt = DataProfilePromptFormatter.for_query_analysis(data_profile)
rprint(data_profile_prompt)


Rows=12; columns=5; complexity=standard; status=ok
Columns:
- Order Date | safe=Order_Date | type=datetime | role=temporal | missing=0.000 | unique=12 | 
min=2024-01-01T00:00:00 | max=2024-12-01T00:00:00 | flags=temporal_like,good_for_temporal_axis | 
prep=normalize_temporal_values | samples=['2024-11-01', '2024-10-01', '2024-01-01', '2024-09-01', '2024-06-01']
- Region | safe=Region | type=categorical | role=dimension | missing=0.000 | unique=4 | 
flags=good_for_grouping,safe_for_color | samples=['East', 'South', 'North', 'North', 'South']
- Sales | safe=Sales | type=numeric | role=measure | missing=0.000 | unique=12 | min=700.0 | max=1710.0 | 
flags=good_for_measure | samples=[1710, 1040, 1200, 1420, 980]
- Profit | safe=Profit | type=numeric | role=measure | missing=0.000 | unique=12 | min=80.0 | max=380.0 | 
flags=good_for_measure | samples=[380, 190, 220, 290, 170]
- Discount | safe=Discount | type=numeric | role=measure | missing=0.000 | unique=12 | min=0.01 | max=0.15 | 
flags=good_for_measure | samples=[0.01, 0.09, 0.05, 0.06, 0.11]
Quality notes: Detected time-like columns: Order Date.

## 3. QueryRequestAnalyzerService

In [12]:
from src.services.query_request_analyzer import QueryRequestAnalyzerService
from src.domain.models import AnalysisRubric

query_analysis = QueryRequestAnalyzerService().invoke(
    query=QUERY,
    user_context=USER_CONTEXT,
    data_profile=data_profile,
    runtime=runtime,
)

visual_judge_requirements = query_analysis.visual_judge_requirements
analysis_rubric = AnalysisRubric(
    focus_areas=[
        query_analysis.analysis_task,
        query_analysis.recommended_chart_family,
        query_analysis.normalized_query,
        "visible chart-grounded findings",
    ],
    strict_visual_only=True,
)

rprint(as_dict(query_analysis))

{
    'normalized_query': 'Show average sales over time by region.',
    'analysis_task': 'trend',
    'recommended_chart_family': 'line',
    'selected_fields': ['Order Date', 'Region', 'Sales'],
    'field_bindings': {
        'x': {
            'field': 'Order Date',
            'role': 'temporal_axis',
            'confidence': 1.0,
            'rationale': "User requested 'over time', and Order Date is the temporal column."
        },
        'y': {
            'field': 'Sales',
            'role': 'measure_axis',
            'confidence': 1.0,
            'rationale': "User requested 'average sales'."
        },
        'color': {
            'field': 'Region',
            'role': 'dimension',
            'confidence': 1.0,
            'rationale': "User requested 'by region', implying a breakdown/grouping."
        }
    },
    'field_mappings': [
        {
            'query_term': 'sales',
            'column_name': 'Sales',
            'confidence': 1.0,
            'rationale': 'Exact match with column name.'
        },
        {
            'query_term': 'time',
            'column_name': 'Order Date',
            'confidence': 1.0,
            'rationale': 'Order Date is the only temporal column available.'
        },
        {
            'query_term': 'region',
            'column_name': 'Region',
            'confidence': 1.0,
            'rationale': 'Exact match with column name.'
        }
    ],
    'aggregation_plan': {'operation': 'mean', 'column': 'Sales', 'group_by': ['Order Date', 'Region']},
    'visual_judge_requirements': {
        'must_be_visible': [
            'X-axis representing time (Order Date)',
            'Y-axis representing average Sales',
            'Distinct lines or colors for each Region'
        ],
        'acceptable_visual_encodings': {},
        'critical_failures': [
            'Missing the temporal axis',
            'Missing the regional breakdown',
            'Using sum instead of average for sales'
        ],
        'yes_no_questions': ['Does the chart show the trend of average sales over time for different regions?']
    },
    'query_variants': [
        {
            'kind': 'canonical',
            'text': 'Show average sales over time by region.',
            'confidence': 1.0,
            'source': 'llm'
        },
        {
            'kind': 'chart_pattern_retrieval',
            'text': 'line chart mean Sales by Order Date and Region',
            'confidence': 0.9,
            'source': 'llm'
        }
    ],
    'chart_answerability': {
        'status': 'answerable_by_chart',
        'reason': 'A multi-line chart can effectively show the average of a measure over time broken down by a 
categorical dimension.'
    },
    'assumptions': ['The user wants a time-series trend analysis.'],
    'ambiguity': {'missing_fields': [], 'notes': [], 'confidence': 1.0},
    'confidence': 1.0
}

## 4. DataPreparationService

In [13]:
from src.services.data_preparation import DataPreparationService

prepared = DataPreparationService().invoke(
    data_path=DATA_PATH.as_posix(),
    data_profile=data_profile,
    request_analysis=query_analysis,
    run_id=RUN_ID,
    runtime=runtime,
)
rprint(as_dict(prepared))


{
    'output_path': 
'D:/programming/projects/ViRAGE/artifacts/manual_service_chain/manual_service_chain_20260524_134601/001_cleaned_dat
a.csv',
    'operations': ['preserve_row_multiplicity', 'safe_column_mapping:1', 'to_datetime:Order Date->Order_Date'],
    'row_count': 12,
    'col_count': 5,
    'column_name_map': {
        'Order Date': 'Order_Date',
        'Region': 'Region',
        'Sales': 'Sales',
        'Profit': 'Profit',
        'Discount': 'Discount'
    },
    'reverse_column_name_map': {
        'Order_Date': 'Order Date',
        'Region': 'Region',
        'Sales': 'Sales',
        'Profit': 'Profit',
        'Discount': 'Discount'
    },
    'original_columns': ['Order Date', 'Region', 'Sales', 'Profit', 'Discount'],
    'safe_columns': ['Order_Date', 'Region', 'Sales', 'Profit', 'Discount'],
    'renamed_column_count': 1
}

 ## 5. VisRAGService

In [16]:
from src.services.visrag import VisRAGService

visrag_result = VisRAGService().invoke(
    query_analysis=query_analysis,
    data_profile=data_profile,
    runtime=runtime,
)
visrag_guidance = visrag_result.generation_guidance

rprint({
    "retrieval_strategy": visrag_result.retrieval_strategy,
    "corpus_status": visrag_result.corpus_status,
    "has_guidance": visrag_guidance.has_guidance,
    "prompt_text": visrag_guidance.prompt_text,
    "retrieved_count_by_type": visrag_result.diagnostics.retrieved_count_by_type,
    "chart_patterns": [as_dict(item) for item in visrag_guidance.chart_patterns],
    "readability_rules": [as_dict(item) for item in visrag_guidance.readability_rules],
    "scale_plot_area_rules": [as_dict(item) for item in visrag_guidance.scale_plot_area_rules],
    "vlm_readability_rules": [as_dict(item) for item in visrag_guidance.vlm_readability_rules],
    "domain_semantics_rules": [as_dict(item) for item in visrag_guidance.domain_semantics_rules],
    "debug_retrieval": as_dict(visrag_result.debug_retrieval),
})


{
    'retrieval_strategy': 'rule_guidance:jsonl:bm25',
    'corpus_status': {
        'enabled': True,
        'documents': 2874,
        'backend': 'jsonl',
        'signature': {
            'backend': 'jsonl',
            'uri': 'D:\\programming\\projects\\ViRAGE\\rag_corpus\\runtime',
            'resolved_path': 'D:/programming/projects/ViRAGE/rag_corpus/runtime/virage_rules.jsonl',
            'exists': True,
            'size_bytes': 2297664,
            'mtime_ns': 1779619391312238800,
            'hash': '325038ff64a407e47037e4c0a92d7154cc525a8f2813d7e8e76f1045c634b49a',
            'cache_key': 
'jsonl:D:/programming/projects/ViRAGE/rag_corpus/runtime/virage_rules.jsonl:1779619391312238800:2297664:325038ff64a
407e47037e4c0a92d7154cc525a8f2813d7e8e76f1045c634b49a',
            'document_count': 2874
        }
    },
    'has_guidance': True,
    'prompt_text': "VisRAG rule guidance:\n- Query analysis is the source of truth for task, selected fields, chart
family, and aggregation. Use retrieved rules only as supporting guidance.\n- Target chart family: line; task: 
trend.\n- Create a choropleth map to show data by region.\n- Create a Choropleth map to show aggregated values by 
region.\n- Make the US and Russia's sales stand out.\n- Add arrows and date labels to show the direction of 
time.\n- Make sure all sales data is visible.\n- Order samples for better readability.\n- Include units of 
measurement on Y-axis labels.",
    'retrieved_count_by_type': {
        'chart_pattern': 2,
        'readability_rule': 2,
        'scale_plot_area_rule': 1,
        'vlm_readability_rule': 2,
        'domain_semantics_rule': 0
    },
    'chart_patterns': [
        {
            'doc_id': 'chart_pattern__from_data_to_viz__choropleth_map__6fa88d1bb4',
            'record_type': 'chart_pattern',
            'title': 'Choropleth Map',
            'prompt_text': 'Create a choropleth map to show data by region.',
            'retrieval_text': 'Choropleth map for geographical data visualization',
            'score': 54.320917,
            'metadata': {
                'applies_when': ['When you want to represent data across geographical regions using colors.'],
                'avoid': [
                    'Avoid using a choropleth map if your data is not geographically distributed or if you do not 
have accurate region shapes.'
                ],
                'chart_family': 'Geographical',
                'guidance': [
                    'Use a choropleth map when you need to show the distribution of quantitative data across 
different geographic areas.'
                ],
                'severity': 'high',
                'source_dataset': 'from_data_to_viz',
                'source_id': 'from_data_to_viz__f90b84565da4',
                'source_weight': 1.0,
                'task': 'Create a choropleth map to visualize data by region.',
                'title': 'Choropleth Map',
                'line_number': 726
            }
        },
        {
            'doc_id': 'chart_pattern__from_data_to_viz__choropleth_map__fd970703e2',
            'record_type': 'chart_pattern',
            'title': 'Choropleth Map',
            'prompt_text': 'Create a Choropleth map to show aggregated values by region.',
            'retrieval_text': 'Choropleth map for geographical data',
            'score': 53.70429,
            'metadata': {
                'applies_when': ['When visualizing data across geographical regions with aggregated values.'],
                'avoid': ['Avoid using solid colors for different categories; use gradients instead.'],
                'chart_family': 'Geographic Visualization',
                'guidance': [
                    'Use a Choropleth map to represent data by varying the color intensity of geographic areas.'
                ],
                'severity': 'high',
                'source_dataset': 'from_data_to_viz',
                'source_id': 'from_data_to_viz__6672956dd5f2',
                'sour

## 6. ChartGeneratorService

In [ ]:
from src.services.chart_generator import ChartGeneratorService

vega_spec = ChartGeneratorService().invoke(
    prepared=prepared,
    runtime=runtime,
    query=QUERY,
    data_profile=data_profile,
    query_request_analysis=query_analysis,
    visrag=visrag_result,
    generation_attempt_number=1,
    max_generation_attempts=1,
    visual_judge_requirements=visual_judge_requirements,
)
as_dict(vega_spec)


## 7. SpecRepairService

In [ ]:
from src.services.spec_repair import SpecRepairService

repaired_spec_json, repair_notes = SpecRepairService().repair(copy.deepcopy(vega_spec.spec_json))
{
    "repair_notes": repair_notes,
    "repaired_spec_json": repaired_spec_json,
}


## 8. SpecValidatorService

In [ ]:
from src.services.spec_validator import SpecValidatorService

spec_validation = SpecValidatorService().invoke(vega_spec)
as_dict(spec_validation)


## 9. VegaLitePlotDrawingService

In [ ]:
from src.services.vegalite_plot_drawing import VegaLitePlotDrawingService
from IPython.display import Image

plot_rendering = VegaLitePlotDrawingService().invoke(
    spec_validation=spec_validation,
    run_id=RUN_ID,
    runtime=runtime,
)
plot_image = plot_rendering.plot_image
Image(filename=plot_image.image_path)


## 10. ScenegraphCheckService

In [ ]:
from src.services.scenegraph_check import ScenegraphCheckService

scenegraph_check = ScenegraphCheckService().invoke(plot_rendering)
as_dict(scenegraph_check)


## 11. EmptyChartCheckService

In [ ]:
from src.services.empty_chart_check import EmptyChartCheckService

empty_chart_check = EmptyChartCheckService().invoke(scenegraph_check)
as_dict(empty_chart_check)


## 12. SpecScoreService

In [ ]:
from src.services.spec_score import SpecScoreService

# Для демонстрации ground truth берём тот же spec. В реальном benchmark сюда передаётся эталонная спецификация.
structural_spec_metric = SpecScoreService().invoke(
    spec_validation=spec_validation,
    ground_truth_spec=spec_validation.validated_spec,
    user_prompt=QUERY,
    empty_chart_check=empty_chart_check,
)
as_dict(structural_spec_metric)


## 13. VisionScoreService

In [ ]:
from src.services.vision_score import VisionScoreService

# Self-mode: один график оценивается относительно запроса. Для VegaChat comparison нужно передать reference_image_path.
vision_score = VisionScoreService().invoke(
    plot_image=plot_image,
    runtime=runtime,
    query_request_analysis=query_analysis,
    user_prompt=QUERY,
)
as_dict(vision_score)


## 14. VLMChartDescriptionService

In [ ]:
from src.services.visual_feedback.vlm_chart_description import VLMChartDescriptionService

vlm_chart_description = VLMChartDescriptionService().invoke(
    plot_image=plot_image,
    runtime=runtime,
)
as_dict(vlm_chart_description)


## 15. ChartFactSummaryService

In [ ]:
from src.services.visual_feedback.chart_fact_summary import ChartFactSummaryService

chart_fact_summary = ChartFactSummaryService().invoke(
    description=vlm_chart_description,
    runtime=runtime,
)
as_dict(chart_fact_summary)


## 16. ChartAnswerJudgeService

In [ ]:
from src.services.visual_feedback.chart_answer_judge import ChartAnswerJudgeService

chart_answer_judge = ChartAnswerJudgeService().invoke(
    query=QUERY,
    chart_facts=chart_fact_summary,
    request_analysis=query_analysis,
    runtime=runtime,
)
as_dict(chart_answer_judge)


## 17. VisualChartJudgeService

In [ ]:
from src.services.visual_feedback.visual_chart_judge import VisualChartJudgeService

visual_chart_judge = VisualChartJudgeService().invoke(
    query=QUERY,
    plot_image=plot_image,
    request_analysis=query_analysis,
    visual_judge_requirements=visual_judge_requirements,
    runtime=runtime,
)
as_dict(visual_chart_judge)


## 18. FeedbackCorpusWriterService

In [ ]:
from src.services.visual_feedback.feedback_corpus_writer import FeedbackCorpusWriterService

feedback_writer = FeedbackCorpusWriterService()
visual_feedback_example = feedback_writer.build_example(
    run_id=RUN_ID,
    attempt_number=1,
    query=QUERY,
    vega_spec=vega_spec,
    rendered_png_path=plot_image.image_path,
    vlm_description=vlm_chart_description,
    chart_facts=chart_fact_summary,
    judge_result=chart_answer_judge,
    request_analysis=query_analysis,
)

# Если нужно реально дописать пример в corpus, раскомментируй строку ниже.
# feedback_corpus_path = feedback_writer.append_to_corpus(visual_feedback_example, runtime=runtime)

as_dict(visual_feedback_example)


## 19. VLMAnalysisService

In [ ]:
from src.services.vlm_analysis import VLMAnalysisService

vlm_analysis = VLMAnalysisService().invoke(
    plot_image=plot_image,
    analysis_rubric=analysis_rubric,
    runtime=runtime,
)
as_dict(vlm_analysis)


## 20. EvaluationSummaryService

In [ ]:
from src.services.evaluation_summary import EvaluationSummaryService
from src.domain.models import SemanticFeedbackLoopSummary, InsightsResult

semantic_summary = SemanticFeedbackLoopSummary(
    enabled=True,
    max_attempts=1,
    attempt_count=1,
    retry_count=0 if visual_chart_judge.retry_recommendation == "accept" else 1,
    accepted=visual_chart_judge.retry_recommendation == "accept",
    accepted_attempt=1 if visual_chart_judge.retry_recommendation == "accept" else None,
    final_status="accepted" if visual_chart_judge.retry_recommendation == "accept" else "failed",
    final_confidence=visual_chart_judge.confidence,
    missing_requirements=visual_chart_judge.missing_requirements,
    improvement_comments=visual_chart_judge.improvement_comments,
)

insights = InsightsResult(final_insights=vlm_analysis.key_findings)

evaluation_summary = EvaluationSummaryService().invoke(
    structural_spec_metric=structural_spec_metric,
    empty_chart_check=empty_chart_check,
    insights=insights,
    technical_status="ok" if spec_validation.is_valid else "failed",
    semantic_status=semantic_summary.final_status,
    semantic_summary=semantic_summary,
    technical_retry_count=0,
    benchmark_scores={
        "spec_score": structural_spec_metric.score,
        "vision_score": vision_score.score,
    },
)
as_dict(evaluation_summary)


## Итог цепочки

In [ ]:
{
    "run_id": RUN_ID,
    "data_path": DATA_PATH.as_posix(),
    "prepared_data_path": prepared.output_path,
    "plot_image_path": plot_image.image_path,
    "spec_valid": spec_validation.is_valid,
    "empty_chart_status": empty_chart_check.empty_chart_status,
    "visual_judge_recommendation": visual_chart_judge.retry_recommendation,
    "visual_judge_confidence": visual_chart_judge.confidence,
    "vlm_analysis_summary": vlm_analysis.summary,
    "artifact_dir": runtime.ensure_run_dir(RUN_ID).as_posix(),
    "token_usage_summary": as_dict(runtime.token_usage_summary()),
}
